In [ ]:
# =========================
# CAPM Portfolio Optimizer (Colab / Jupyter)
# - Data: yfinance (no CSV)
# - Method: Mean-Variance (Monte Carlo Efficient Frontier)
# - Output: Efficient Frontier + (optional) CML + Best (Max Sharpe) portfolio
# =========================

# ---- 0) Install (Colab only) ----
!pip -q install yfinance

# ---- 1) Imports ----
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# ---- 2) Config (EDIT HERE) ----
# Korea tickers examples:
# Samsung Electronics: 005930.KS
# SK hynix:           000660.KS
# NAVER:              035420.KS

TICKERS = ["005930.KS", "000660.KS", "035420.KS"]

START_DATE = "2018-01-01"   # backtest data start
END_DATE = None            # None = latest

TRADING_DAYS = 252
RISK_FREE_RATE = 0.02      # annual (e.g., 2%) - adjust later if you want
N_PORTFOLIOS = 30000       # Monte Carlo samples

# Save figures here (GitHub-friendly)
# In your repo: put notebook in /notebooks and save to ../assets.
# In Colab: easiest is saving to ./assets and upload that folder to GitHub.
SAVE_DIR = "assets"        # change to "../assets" when running inside repo notebooks/


# ---- 3) Helpers ----
def download_prices(tickers: list[str], start: str, end: str | None) -> pd.DataFrame:
    """
    Download adjusted close prices for given tickers.
    """
    data = yf.download(tickers, start=start, end=end, auto_adjust=False, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        prices = data["Adj Close"].copy()
    else:
        # If only one ticker, yfinance sometimes returns a single-index frame
        prices = data[["Adj Close"]].rename(columns={"Adj Close": tickers[0]})

    prices = prices.dropna(how="all")
    prices = prices.dropna(axis=1, how="any")  # drop tickers with missing series
    return prices


def annualize_returns_cov(prices: pd.DataFrame, trading_days: int = 252) -> tuple[pd.Series, pd.DataFrame, pd.DataFrame]:
    """
    Compute daily returns, annualized mean returns and annualized covariance.
    """
    rets = prices.pct_change().dropna()
    mu = rets.mean() * trading_days
    cov = rets.cov() * trading_days
    return mu, cov, rets


def simulate_portfolios(mu: pd.Series, cov: pd.DataFrame, rf: float, n: int, seed: int = 42):
    """
    Monte Carlo simulation of random portfolios.
    Returns:
      results: (n, 3) array: [vol, ret, sharpe]
      weights: (n, k) array weights
    """
    rng = np.random.default_rng(seed)
    k = len(mu)

    weights = rng.random((n, k))
    weights = weights / weights.sum(axis=1, keepdims=True)

    # Portfolio returns: w @ mu
    port_rets = weights @ mu.values

    # Portfolio variance: diag(W Σ W^T)
    # Efficient computation:
    port_vars = np.einsum("ij,jk,ik->i", weights, cov.values, weights)
    port_vols = np.sqrt(port_vars)

    sharpe = (port_rets - rf) / port_vols

    results = np.column_stack([port_vols, port_rets, sharpe])
    return results, weights


def find_tangency(results: np.ndarray, weights: np.ndarray):
    """
    Find max Sharpe portfolio index and return its stats + weights.
    """
    best_idx = int(np.nanargmax(results[:, 2]))
    best_vol, best_ret, best_sharpe = results[best_idx]
    best_w = weights[best_idx]
    return best_idx, best_w, best_ret, best_vol, best_sharpe


def plot_efficient_frontier(results: np.ndarray, best_vol: float, best_ret: float, title: str, save_path: str | None = None):
    """
    Plot the Monte Carlo efficient frontier.
    """
    plt.figure(figsize=(9, 5.5))
    sc = plt.scatter(results[:, 0], results[:, 1], c=results[:, 2], s=6, alpha=0.85)
    plt.colorbar(sc, label="Sharpe Ratio")
    plt.scatter(best_vol, best_ret, marker="*", s=280)
    plt.xlabel("Volatility (Annualized)")
    plt.ylabel("Return (Annualized)")
    plt.title(title)
    plt.grid(True, alpha=0.25)
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_cml(rf: float, best_ret: float, best_vol: float, max_x: float, title: str, save_path: str | None = None):
    """
    Plot Capital Market Line using the tangency portfolio.
    """
    # CML: E[R] = rf + ( (E[R_tan]-rf)/σ_tan ) * σ
    slope = (best_ret - rf) / best_vol
    x = np.linspace(0, max_x, 200)
    y = rf + slope * x

    plt.figure(figsize=(9, 5.5))
    plt.plot(x, y)
    plt.scatter([0, best_vol], [rf, best_ret], s=80)
    plt.xlabel("Volatility (Annualized)")
    plt.ylabel("Return (Annualized)")
    plt.title(title)
    plt.grid(True, alpha=0.25)
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


# ---- 4) Run ----
os.makedirs(SAVE_DIR, exist_ok=True)

prices = download_prices(TICKERS, START_DATE, END_DATE)

print("Downloaded tickers:", list(prices.columns))
print("Date range:", prices.index.min().date(), "to", prices.index.max().date())
display(prices.tail())

mu, cov, daily_returns = annualize_returns_cov(prices, TRADING_DAYS)

print("\nAnnualized Expected Returns (mu):")
display(mu.to_frame("mu").T)

print("\nAnnualized Covariance Matrix (cov):")
display(cov)

results, weights = simulate_portfolios(mu, cov, RISK_FREE_RATE, N_PORTFOLIOS, seed=42)
best_idx, best_w, best_ret, best_vol, best_sharpe = find_tangency(results, weights)

print("\n=== Best (Max Sharpe) Portfolio ===")
for t, w in zip(prices.columns, best_w):
    print(f"- {t}: {w:.4f}")
print(f"Return: {best_ret:.4f} | Vol: {best_vol:.4f} | Sharpe: {best_sharpe:.4f}")

# ---- 5) Plots + Save images ----
frontier_path = os.path.join(SAVE_DIR, "efficient_frontier.png")
plot_efficient_frontier(
    results=results,
    best_vol=best_vol,
    best_ret=best_ret,
    title="Efficient Frontier (Monte Carlo)",
    save_path=frontier_path
)

# Optional: CML
cml_path = os.path.join(SAVE_DIR, "cml.png")
max_x = max(np.percentile(results[:, 0], 99), best_vol)  # reasonable x-range
plot_cml(
    rf=RISK_FREE_RATE,
    best_ret=best_ret,
    best_vol=best_vol,
    max_x=max_x,
    title="Capital Market Line (CML)",
    save_path=cml_path
)

print(f"\nSaved plots to: {os.path.abspath(SAVE_DIR)}")
print(" -", frontier_path)
print(" -", cml_path)

# ---- 6) (Optional) Export weights to CSV for README / report ----
weights_df = pd.DataFrame({"Ticker": list(prices.columns), "Weight": best_w})
weights_csv = os.path.join(SAVE_DIR, "best_portfolio_weights.csv")
weights_df.to_csv(weights_csv, index=False, encoding="utf-8-sig")
print("Saved weights to:", weights_csv)